In [1]:
import openai, json

client = openai.OpenAI()
messages = []

In [2]:
import requests
import json

BASE_URL = "https://nomad-movies.nomadcoders.workers.dev"

def get_popular_movies():
    """인기 영화 목록을 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

def get_movie_details(id):
    """ID로 영화 상세 정보를 조회합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{id}")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

def get_movie_credits(id):
    """영화의 출연진 및 제작진 정보를 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{id}/credits")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

def get_movie_similar(id):
    """영화의 유사 영화 목록을 반환합니다."""
    resp = requests.get(f"{BASE_URL}/movies/{id}/similar")
    resp.raise_for_status()
    return json.dumps(resp.json(), ensure_ascii=False)

FUNCTION_MAP = {
    "get_popular_movies": get_popular_movies,
    "get_movie_details": get_movie_details,
    "get_movie_credits": get_movie_credits,
    "get_movie_similar": get_movie_similar
}

In [3]:
TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "get_popular_movies",
            "description": "Get a list of popular movies."
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_details",
            "description": "Get the details of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_credits",
            "description": "Get the credits of a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_movie_similar",
            "description": "Get a list of similar movies to a specific movie by its ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "id": {
                        "type": "string",
                        "description": "The ID of the movie."
                    }
                },
                "required": ["id"]
            }
        }
    }
]

In [4]:
from openai.types.chat import ChatCompletionMessage

def process_ai_response(message: ChatCompletionMessage):
    if message.tool_calls:
        messages.append(
            {
                "role":"assistant", 
                "content":message.content or "",
                "tool_calls":[
                    {
                        "id":tool_call.id,
                        "type":"function",
                        "function":{
                            "name":tool_call.function.name,
                            "arguments":tool_call.function.arguments
                        },
                    } for tool_call in message.tool_calls
                ],
            }
        )

        for tool_call in message.tool_calls:
            function_name = tool_call.function.name
            arguments = tool_call.function.arguments

            print(f"Calling function: {function_name} with {arguments}")

            try:
                arguments = json.loads(arguments)
            except json.JSONDecodeError:
                arguments = {}

            function_to_run = FUNCTION_MAP.get(function_name)

            result = function_to_run(**arguments)
            
            # print(f"Ran {function_name} with args {arguments} for a result of {result}")
            print(f"Ran {function_name} with args {arguments}")

            messages.append({
                "role":"tool",
                "tool_call_id":tool_call.id,
                "name":function_name,
                "content":result,
            })
        Call_ai()
    else :
        messages.append({"role":"assistant", "content":message.content})
        print(f"AI : {message.content}")


def Call_ai():
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=messages,
        tools=TOOLS,
    )
    process_ai_response(response.choices[0].message)

In [5]:
while True:
    message = input("Send a message to the LLM...")
    if message == "quit" or message == "q":
        break
    else:
        messages.append({"role":"user", "content":message})
        print(f"User : {message}")
        Call_ai()

User : 지금 인기 있는 영화 알려줘
Calling function: get_popular_movies with {}
Ran get_popular_movies with args {}
AI : 현재 인기 있는 영화 목록입니다:

1. **Shelter**
   - 개요: 한 남자가 자발적으로 망명 중인 외딴 섬에서 폭풍우로 고립된 소녀를 구출하면서 과거의 적들로부터 그녀를 보호하기 위해 은둔에서 벗어나게 되는 이야기입니다.
   - 개봉일: 2026년 1월 28일
   - 평점: 6.96
   - ![Shelter](https://image.tmdb.org/t/p/w780/buPFnHZ3xQy6vZEHxbHgL1Pc6CR.jpg)

2. **Mercy**
   - 개요: 가까운 미래, 한 탐정이 아내를 살해한 혐의로 재판을 받으며 자신의 무죄를 증명하기 위해 AI 판사에게 90분의 시간을 주어야 하는 이야기입니다.
   - 개봉일: 2026년 1월 20일
   - 평점: 7.1
   - ![Mercy](https://image.tmdb.org/t/p/w780/pyok1kZJCfyuFapYXzHcy7BLlQa.jpg)

3. **The Orphans**
   - 개요: 유년 시절 친구였던 두 남자, 한 명은 경찰, 다른 한 명은 범죄자에 가까워진 그들이 사망한 첫사랑의 딸을 구하기 위해 함께 힘을 합치는 이야기입니다.
   - 개봉일: 2025년 8월 20일
   - 평점: 6.18
   - ![The Orphans](https://image.tmdb.org/t/p/w780/hP7mjZr2SVfjAorlRHTdV1XZmHY.jpg)

4. **The Bluff**
   - 개요: 외딴 섬에서 평온한 삶을 살던 전 해적이 복수심에 불타는 전직 선장과 마주치면서 과거의 혈투를 되돌리게 되는 이야기입니다.
   - 개봉일: 2026년 2월 17일
   - 평점: 5.83
   - ![The Bluff](https://image.tmdb.org/t/p/w780/soj